In [ ]:
# Hair and Scalp Diagnosis System for Google Colab
# Install required libraries
!pip install tensorflow keras pillow opencv-python numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from PIL import Image
import cv2
from google.colab import files
import io


In [ ]:
diagnosis_system_instance_for_saving = HairScalpDiagnosis()
diagnosis_system_instance_for_saving.model.save("model.h5")

Loading pre-trained model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from google.colab import files
files.download("model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:


# Dictionary with hair/scalp conditions, causes, precautions, and treatments
HAIR_CONDITIONS = {

    'Dandruff': {
        'description': 'White flakes on scalp and hair',
        'causes': [
            'Dry scalp',
            'Malassezia fungus overgrowth',
            'Poor hygiene',
            'Stress and hormonal imbalance'
        ],
        'precautions': [
            'Use anti-dandruff shampoo containing zinc pyrithione or ketoconazole',
            'Wash hair regularly (every alternate day)',
            'Avoid tight hairstyles',
            'Manage stress through exercise and meditation',
            'Maintain proper diet with omega-3 fatty acids',
            'Avoid harsh chemical treatments'
        ],
        'treatment': [
            'Apply medicated shampoo for 2-3 weeks',
            'Use scalp moisturizer or oil',
            'Consider dermatologist consultation if persistent'
        ]
    },
    'Hair Loss': {
        'description': 'Excessive hair shedding',
        'causes': [
            'Genetics (androgenetic alopecia)',
            'Hormonal imbalance',
            'Nutritional deficiency',
            'Stress and anxiety',
            'Poor hair care practices',
            'Underlying medical conditions'
        ],
        'precautions': [
            'Avoid tight braiding and hairstyles',
            'Minimize heat styling',
            'Use sulfate-free shampoos',
            'Eat protein-rich foods',
            'Take biotin and iron supplements',
            'Manage stress effectively',
            'Massage scalp regularly'
        ],
        'treatment': [
            'Consult a dermatologist',
            'Consider minoxidil (Rogaine) treatment',
            'Use fortifying hair products',
            'Hair growth supplements',
            'PRP therapy or low-level laser therapy (if recommended)'
        ]
    },



    'Alopecia Areata': {
        'description': 'Patchy hair loss in circular areas',
        'causes': [
            'Autoimmune condition',
            'Genetic predisposition',
            'Severe stress',
            'Emotional trauma'
        ],
        'precautions': [
            'Manage stress through therapy',
            'Avoid pulling/picking at hair',
            'Use gentle hair handling',
            'Wear protective styles if desired',
            'Maintain overall health',
            'Get adequate sleep'
        ],
        'treatment': [
            'Topical corticosteroid injections',
            'Topical minoxidil',
            'Dermatologist consultation essential',
            'Immunotherapy treatments',
            'Wigs or hairpieces as cosmetic solution'
        ]
    },
    'White Hair': {
    'description': 'Loss of natural hair pigment leading to gray or white strands',
    'causes': [
        'Genetic factors',
        'Aging process',
        'Vitamin B12 deficiency',
        'Iron or copper deficiency',
        'Thyroid disorders',
        'Oxidative stress',
        'Smoking'
    ],
    'precautions': [
        'Maintain a balanced diet rich in vitamins and minerals',
        'Avoid excessive chemical treatments',
        'Limit heat styling tools',
        'Manage stress effectively',
        'Quit smoking',
        'Regular scalp care'
    ],
    'treatment': [
        'Nutritional supplements if deficiency is present',
        'Treat underlying medical conditions',
        'Use mild, ammonia-free hair dyes if desired',
        'Antioxidant-rich diet',
        'Dermatologist consultation for early graying'
    ]
},

'Head Lice': {
    'description': 'Infestation of the scalp by tiny parasitic insects that feed on blood',
    'causes': [
        'Direct head-to-head contact',
        'Sharing personal items (combs, hats, towels)',
        'Close contact in crowded environments'
    ],
    'precautions': [
        'Avoid sharing personal hair items',
        'Regularly wash and inspect hair',
        'Maintain scalp hygiene',
        'Tie long hair in public places',
        'Educate close contacts if infestation occurs'
    ],
    'treatment': [
        'Medicated anti-lice shampoos (permethrin or pyrethrin)',
        'Wet combing with fine-toothed lice comb',
        'Repeat treatment after 7–10 days',
        'Wash clothing and bedding in hot water',
        'Dermatologist consultation for resistant cases'
    ]
}



}

In [ ]:
HAIR_CONDITIONS = {

    'Dandruff': {
        'description': 'White flakes on scalp and hair',
        'causes': [
            'Dry scalp',
            'Malassezia fungus overgrowth',
            'Poor hygiene',
            'Stress and hormonal imbalance'
        ],
        'precautions': [
            'Use anti-dandruff shampoo containing zinc pyrithione or ketoconazole',
            'Wash hair regularly (every alternate day)',
            'Avoid tight hairstyles',
            'Manage stress through exercise and meditation',
            'Maintain proper diet with omega-3 fatty acids',
            'Avoid harsh chemical treatments'
        ],
        'treatment': [
            'Apply medicated shampoo for 2-3 weeks',
            'Use scalp moisturizer or oil',
            'Consider dermatologist consultation if persistent'
        ]
    }}

In [ ]:
class HairScalpDiagnosis:
    def __init__(self):
        """Initialize the diagnosis system"""
        self.condition_labels = list(HAIR_CONDITIONS.keys()) # Moved this line up
        self.model = self._load_model()

    def _load_model(self):
        """Load pre-trained model"""
        print("Loading pre-trained model...")
        model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

        # Add custom layers for classification
        x = tf.keras.layers.GlobalAveragePooling2D()(model.output)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        output = tf.keras.layers.Dense(len(self.condition_labels), activation='softmax')(x)

        final_model = tf.keras.Model(inputs=model.input, outputs=output)
        final_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

        return final_model

    def preprocess_image(self, img_path):
        """Preprocess image for model"""
        img = image.load_img(img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
        return img_array

    def diagnose(self, img_path):
        """Diagnose hair/scalp condition"""
        # Preprocess image
        processed_img = self.preprocess_image(img_path)

        # Make prediction
        predictions = self.model.predict(processed_img, verbose=0)
        predicted_class_idx = np.argmax(predictions[0])
        confidence = predictions[0][predicted_class_idx]
        predicted_condition = self.condition_labels[predicted_class_idx]

        return predicted_condition, confidence, predictions[0]

    def display_diagnosis(self, img_path, condition, confidence):
        """Display diagnosis with image and recommendations"""
        # Load and display image
        img = Image.open(img_path)
        plt.figure(figsize=(12, 8))

        # Display image
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title('Uploaded Image', fontsize=14, fontweight='bold')
        plt.axis('off')

        # Display diagnosis info
        plt.subplot(1, 2, 2)
        plt.axis('off')

        condition_info = HAIR_CONDITIONS[condition]

        diagnosis_text = f"""
DIAGNOSIS REPORT
{'='*50}

CONDITION: {condition}
Confidence: {confidence*100:.2f}%

DESCRIPTION:
{condition_info['description']}

CAUSES:
"""
        if isinstance(condition_info['causes'], list):
            for cause in condition_info['causes']:
                diagnosis_text += f"\n• {cause}"
        else:
            diagnosis_text += f"\n• {condition_info['causes']}"

        diagnosis_text += f"\n\nPRECAUTIONS:\n"
        for precaution in condition_info['precautions']:
            diagnosis_text += f"\n✓ {precaution}"

        diagnosis_text += f"\n\nRECOMMENDED TREATMENT:\n"
        if isinstance(condition_info['treatment'], list):
            for treatment in condition_info['treatment']:
                diagnosis_text += f"\n→ {treatment}"
        else:
            diagnosis_text += f"\n→ {condition_info['treatment']}"

        plt.text(0.05, 0.95, diagnosis_text, transform=plt.gca().transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plt.tight_layout()
        plt.show()

        return condition_info

    def get_detailed_report(self, condition):
        """Get detailed report for a condition"""
        return HAIR_CONDITIONS.get(condition, None)

# Main function to run the app
def main():
    """Main function to run hair diagnosis"""
    print("\n" + "="*60)
    print("HAIR AND SCALP DIAGNOSIS SYSTEM")
    print("="*60 + "\n")

    # Initialize diagnosis system
    diagnosis_system = HairScalpDiagnosis()

    # Upload image
    print("Please upload your hair/scalp image...")
    uploaded = files.upload()

    if len(uploaded) == 0:
        print("No file uploaded!")
        return

    # Get uploaded file name
    img_filename = list(uploaded.keys())[0]

    # Run diagnosis
    print(f"\nAnalyzing your image: {img_filename}")
    print("Please wait...\n")

    condition, confidence, all_predictions = diagnosis_system.diagnose(img_filename)

    # Display diagnosis
    diagnosis_system.display_diagnosis(img_filename, condition, confidence)

    # Print detailed diagnosis
    print("\n" + "="*60)
    print("DETAILED DIAGNOSIS REPORT")
    print("="*60)
    info = diagnosis_system.get_detailed_report(condition)

    print(f"\n✓ IDENTIFIED CONDITION: {condition}")
    print(f"✓ CONFIDENCE LEVEL: {confidence*100:.2f}%\n")

    print(f"Description: {info['description']}\n")

    print("Possible Causes:")
    if isinstance(info['causes'], list):
        for i, cause in enumerate(info['causes'], 1):
            print(f"  {i}. {cause}")
    else:
        print(f"  • {info['causes']}")

    print("\nRecommended Precautions:")
    for i, precaution in enumerate(info['precautions'], 1):
        print(f"  {i}. {precaution}")

    print("\nTreatment Options:")
    if isinstance(info['treatment'], list):
        for i, treatment in enumerate(info['treatment'], 1):
            print(f"  {i}. {treatment}")
    else:
        print(f"  • {info['treatment']}")

    print("\n" + "="*60)
    print("⚠️  IMPORTANT: Please consult a dermatologist for professional")
    print("diagnosis and treatment. This is an AI-assisted analysis only.")
    print("="*60 + "\n")

# Run the app
if __name__ == "__main__":
    main()


HAIR AND SCALP DIAGNOSIS SYSTEM



NameError: name 'HAIR_CONDITIONS' is not defined

In [ ]:
!pip install pillow numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls


In [ ]:
from PIL import Image
import numpy as np

DELIMITER = "#####"

def encode_image(image_path, secret_text, output_path):
    img = Image.open(image_path).convert("RGB")
    data = np.array(img)

    secret_text += DELIMITER
    binary = ''.join(format(ord(c), '08b') for c in secret_text)

    flat_pixels = data.flatten()

    if len(binary) > len(flat_pixels):
        raise ValueError("Text too long to hide in this image")

    # Fix: Use 254 (0b11111110) to clear the LSB for uint8 to avoid OverflowError
    for i in range(len(binary)):
        flat_pixels[i] = (flat_pixels[i] & 254) | int(binary[i])

    encoded_img = flat_pixels.reshape(data.shape)
    Image.fromarray(encoded_img).save(output_path)

    print("✅ Encoding complete. Saved as:", output_path)


# Example usage
encode_image(
    image_path="/content/img1.jpeg", # Changed to an existing file
    secret_text="Diagnosis: Dandruff | Causes: Fungal | Precaution: Ketoconazole shampoo",
    output_path="encoded.png"
)


In [ ]:
from PIL import Image
import numpy as np

DELIMITER = "#####"

def decode_image(image_path):
    img = Image.open(image_path)
    data = np.array(img)

    flat_pixels = data.flatten()

    binary = ""
    for pixel in flat_pixels:
        binary += str(pixel & 1)

    chars = []
    for i in range(0, len(binary), 8):
        byte = binary[i:i+8]
        chars.append(chr(int(byte, 2)))
        if ''.join(chars).endswith(DELIMITER):
            break

    message = ''.join(chars).replace(DELIMITER, "")
    return message


# Example usage
hidden_text = decode_image("encoded.png")
print("🔎 Report:", hidden_text)